# 🧹 Data Cleaning & Feature Engineering
**CSC 3221 — Introduction to Data Science | ICT University**

**Input:** `output/master_transactions.csv` + `output/demographics_template.csv`

**Output:** `2_Data_Cleaning/cleaned_data.csv` — one row per user, ready for EDA and modeling

### What this notebook covers
1. Load and assess data quality
2. Clean: handle missing values, fix types, remove duplicates
3. Feature engineering: derive meaningful variables per user
4. Merge with demographics
5. Handle outliers
6. Encode categorical variables
7. Export cleaned dataset

## ⚙️ Step 1 — Imports & Paths

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from IPython.display import display

# Paths
NOTEBOOK_DIR = Path().resolve().parent  # one level up from 2_Data_Cleaning/
OUTPUT_DIR   = NOTEBOOK_DIR / 'output'
CLEAN_DIR    = NOTEBOOK_DIR / '2_Data_Cleaning'
CLEAN_DIR.mkdir(exist_ok=True)

MASTER_FILE  = OUTPUT_DIR / 'master_transactions.csv'
DEMO_FILE    = OUTPUT_DIR / 'demographics_template.csv'
CLEANED_FILE = CLEAN_DIR  / 'cleaned_data.csv'

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

print('✅ Ready!')


✅ Ready!


## 📂 Step 2 — Load Raw Data

We load two datasets:
- **master_transactions.csv** — one row per transaction, all users combined
- **demographics_template.csv** — one row per user with their survey answers

We immediately check shapes and column types. This is called a **data audit** — the first thing any data scientist does before touching the data.

In [2]:
df_tx   = pd.read_csv(MASTER_FILE, parse_dates=['Date'])
df_demo = pd.read_csv(DEMO_FILE)

print('=== Transactions ===')
print(f'Shape : {df_tx.shape[0]:,} rows × {df_tx.shape[1]} columns')
print(f'Users : {df_tx["UserId"].nunique()}')
print(f'Date range: {df_tx["Date"].min().date()} to '
      f'{df_tx["Date"].max().date()}')
print()
display(df_tx.head(3))

print('\n=== Demographics ===')
print(f'Shape : {df_demo.shape}')
display(df_demo.head(3))


=== Transactions ===
Shape : 7,797 rows × 10 columns
Users : 16
Date range: 2021-02-10 to 2026-12-03



,UserId,Date,Time,Operator,Transaction_type,Direction,Amount,Currency,New_balance,Anonymized_Content
0,USER_1,2023-10-22,00:38:30,MobileMoney,paiement,OUT,200.0,XAF,594.0,Votre paiement de 200 XAF a MTNC BUNDLES_FORFA...
1,USER_1,2023-10-22,01:26:42,MobileMoney,paiement,OUT,200.0,XAF,394.0,Votre paiement de 200 XAF a MTNC BUNDLES_FORFA...
2,USER_1,2023-11-21,17:02:53,MobileMoney,transfert,IN,100.0,XAF,494.0,An adjustment has been made and 100 XAF has be...



=== Demographics ===
Shape : (16, 10)


,UserId,Age_range,Gender,Occupation,Education_level,Monthly_income_range,Geographic_zone,Household_size,Primary_MM_use,Smartphone_ownership
0,USER_1,46-60,F,Salaried employee,University – Postgraduate,300001-500000,Suburban,7+,Both personal and business,Yes
1,USER_2,46-60,M,Salaried employee,University – Postgraduate,300001-500000,Suburban,7+,Both personal and business,Yes
2,USER_3,46-60,F,Salaried employee,University – Postgraduate,300001-500000,Suburban,7+,Both personal and business,Yes


## 🔍 Step 3 — Data Quality Assessment

Before cleaning, we document every problem we find. This is required for the **Data Cleaning Report** and shows the examiner you understand your data's limitations.

In [3]:
print('=== Missing Values (Transactions) ===')
missing_tx = df_tx.isnull().sum()
missing_pct = (missing_tx / len(df_tx) * 100).round(2)
quality_tx = pd.DataFrame({
    'Missing Count': missing_tx,
    'Missing %':     missing_pct,
    'Dtype':         df_tx.dtypes,
})
display(quality_tx[quality_tx['Missing Count'] > 0])

print('\n=== Missing Values (Demographics) ===')
missing_demo = df_demo.isnull().sum() + (df_demo == '').sum()
missing_demo_pct = (missing_demo / len(df_demo) * 100).round(2)
quality_demo = pd.DataFrame({
    'Missing Count': missing_demo,
    'Missing %':     missing_demo_pct,
})
display(quality_demo)

print('\n=== Duplicate Transactions ===')
dups = df_tx.duplicated().sum()
print(f'{dups} exact duplicate rows found')

print('\n=== Transaction Type Distribution ===')
display(df_tx['Transaction_type'].value_counts())

print('\n=== Direction Distribution ===')
display(df_tx['Direction'].value_counts())


=== Missing Values (Transactions) ===


,Missing Count,Missing %,Dtype
Amount,25,0.32,float64
Currency,25,0.32,str
New_balance,78,1.00,float64



=== Missing Values (Demographics) ===


,Missing Count,Missing %
UserId,0,0.00
Age_range,2,12.50
Gender,2,12.50
Occupation,2,12.50
Education_level,2,12.50
Monthly_income_range,6,37.50
Geographic_zone,2,12.50
Household_size,2,12.50
Primary_MM_use,2,12.50
Smartphone_ownership,3,18.75



=== Duplicate Transactions ===
0 exact duplicate rows found

=== Transaction Type Distribution ===


Transaction_type
transfert       4139
retrait         1051
rechargement     724
depot            713
transaction      655
paiement         464
autre             51
Name: count, dtype: int64


=== Direction Distribution ===


Direction
OUT        6571
IN         1175
unknown      51
Name: count, dtype: int64

## 🧼 Step 4 — Clean the Transaction Data

We apply targeted fixes for each issue found above.

**Decision log** — justify every decision:
- **Duplicate removal:** Exact duplicates are kept only once. A message cannot legitimately appear twice.
- **Missing Amount:** We do NOT drop rows with missing amounts. The message still tells us a transaction happened and its type — useful for frequency features. We only drop when both Amount AND New_balance are missing.
- **'autre' transactions:** Kept in the dataset but flagged. Dropping them would bias frequency counts downward.

In [4]:
df = df_tx.copy()

# ── 1. Remove exact duplicates ────────────────────────────────────────
before = len(df)
df = df.drop_duplicates()
print(f'Removed {before - len(df)} duplicate rows')

# ── 2. Standardise types ──────────────────────────────────────────────
df['Amount']      = pd.to_numeric(df['Amount'], errors='coerce')
df['New_balance'] = pd.to_numeric(df['New_balance'], errors='coerce')
df['Date']        = pd.to_datetime(df['Date'], format='mixed', dayfirst=True, errors='coerce')

# ── 3. Drop rows where both Amount AND New_balance are missing ─────────
# Rationale: these rows give us no financial information at all.
before = len(df)
df = df.dropna(subset=['Amount', 'New_balance'], how='all')
print(f'Removed {before - len(df)} rows with no amount and no balance')

# ── 4. Drop invalid dates ─────────────────────────────────────────────
before = len(df)
df = df.dropna(subset=['Date'])
print(f'Removed {before - len(df)} rows with unparseable dates')

# ── 5. Add useful time columns ────────────────────────────────────────
df['YearMonth']    = df['Date'].dt.to_period('M')
df['DayOfWeek']    = df['Date'].dt.dayofweek          # 0=Mon, 6=Sun
df['IsWeekend']    = df['DayOfWeek'].isin([5, 6])
df['Hour']         = pd.to_datetime(
                         df['Time'], format='%H:%M:%S', errors='coerce'
                     ).dt.hour

print(f'\nClean dataset shape: {df.shape}')
display(df.dtypes)


Removed 0 duplicate rows
Removed 25 rows with no amount and no balance
Removed 0 rows with unparseable dates

Clean dataset shape: (7772, 14)


UserId                           str
Date                  datetime64[us]
Time                             str
Operator                         str
Transaction_type                 str
Direction                        str
Amount                       float64
Currency                         str
New_balance                  float64
Anonymized_Content               str
YearMonth                  period[M]
DayOfWeek                      int32
IsWeekend                       bool
Hour                         float64
dtype: object

## ⚙️ Step 5 — Feature Engineering

**What is feature engineering?** It's creating new variables from existing ones that are more informative for a model. A model cannot easily learn from 1,000 raw transaction rows for one person — but it can learn easily from 'this person makes 15 transactions/month averaging 8,000 FCFA each.'

We aggregate from **transaction level** (one row = one transaction) to **user level** (one row = one person). This is the dataset we'll model.

Each feature has a justification — this is what you explain in the viva:

In [5]:
# ── Compute per-user features ─────────────────────────────────────────

def user_features(grp: pd.DataFrame) -> pd.Series:
    """
    Takes all transactions for one user and returns a single row
    of summary features.
    """
    total_tx      = len(grp)
    in_tx         = grp[grp['Direction'] == 'IN']
    out_tx        = grp[grp['Direction'] == 'OUT']

    # Date span — how many months of data does this user have?
    date_min      = grp['Date'].min()
    date_max      = grp['Date'].max()
    months_active = max(1,
        ((date_max.year - date_min.year) * 12
         + date_max.month - date_min.month + 1)
    )

    # Transaction frequency
    tx_per_month  = total_tx / months_active

    # Amount statistics — only on non-null amounts
    amounts = grp['Amount'].dropna()
    avg_amount = amounts.mean()   if len(amounts) > 0 else np.nan
    med_amount = amounts.median() if len(amounts) > 0 else np.nan
    std_amount = amounts.std()    if len(amounts) > 1 else 0.0
    total_in   = in_tx['Amount'].sum()
    total_out  = out_tx['Amount'].sum()

    # Send/Receive ratio
    # > 1 means sends more than receives (more outgoing behaviour)
    # < 1 means receives more
    n_in  = len(in_tx)
    n_out = len(out_tx)
    send_receive_ratio = (n_out / n_in) if n_in > 0 else np.nan

    # Weekend activity proportion
    weekend_ratio = grp['IsWeekend'].mean()

    # Balance stats — proxy for financial stability
    balances = grp['New_balance'].dropna()
    avg_balance = balances.mean()   if len(balances) > 0 else np.nan
    min_balance = balances.min()    if len(balances) > 0 else np.nan
    max_balance = balances.max()    if len(balances) > 0 else np.nan

    # Days since last transaction (recency)
    last_date = date_max
    reference = grp['Date'].max()  # use within-dataset max as reference
    recency_days = (reference - last_date).days

    # Transaction type diversity
    n_types = grp['Transaction_type'].nunique()

    # Transaction velocity: slope of monthly transaction count over time
    # Positive = increasing activity, negative = declining activity
    monthly_counts = (
        grp.groupby('YearMonth').size().reset_index(name='count')
    )
    if len(monthly_counts) >= 2:
        x = np.arange(len(monthly_counts))
        velocity = float(np.polyfit(x, monthly_counts['count'], 1)[0])
    else:
        velocity = 0.0

    # Operator (take first — same user usually has one per file)
    operator = grp['Operator'].mode().iloc[0] \
               if len(grp['Operator'].mode()) > 0 else 'Unknown'

    return pd.Series({
        'total_transactions':  total_tx,
        'months_active':       months_active,
        'tx_per_month':        round(tx_per_month, 2),
        'avg_amount':          round(avg_amount, 2),
        'median_amount':       round(med_amount, 2),
        'std_amount':          round(std_amount, 2),
        'total_amount_in':     round(total_in,  2),
        'total_amount_out':    round(total_out, 2),
        'n_in':                n_in,
        'n_out':               n_out,
        'send_receive_ratio':  round(send_receive_ratio, 3),
        'weekend_ratio':       round(weekend_ratio, 3),
        'avg_balance':         round(avg_balance, 2),
        'min_balance':         round(min_balance, 2),
        'max_balance':         round(max_balance, 2),
        'recency_days':        recency_days,
        'n_tx_types':          n_types,
        'tx_velocity':         round(velocity, 4),
        'primary_operator':    operator,
    })


print('Computing per-user features...')
df_users = df.groupby('UserId').apply(user_features).reset_index()
print(f'✅ Feature matrix shape: {df_users.shape}')
print(f'   {df_users.shape[0]} users × {df_users.shape[1]} features')
display(df_users.head())


Computing per-user features...
✅ Feature matrix shape: (16, 20)
   16 users × 20 features


,UserId,total_transactions,months_active,tx_per_month,avg_amount,median_amount,std_amount,total_amount_in,total_amount_out,n_in,n_out,send_receive_ratio,weekend_ratio,avg_balance,min_balance,max_balance,recency_days,n_tx_types,tx_velocity,primary_operator
0,USER_1,147,30,4.90,9872.82,1000.0,17014.06,779704.0,671600.0,45,102,2.267,0.224,14101.95,25.00,90419.0,0,4,0.2187,MobileMoney
1,USER_10,795,36,22.08,3837.41,500.0,23813.61,1382350.0,1668393.0,111,684,6.162,0.282,5667.87,2.00,450196.0,0,4,-0.1140,MobileMoney
2,USER_11,315,24,13.12,13478.13,5500.0,21315.02,2014234.0,2231377.0,127,188,1.480,0.289,26025.06,0.00,231246.0,0,4,-0.5726,MobileMoney
3,USER_12,104,13,8.00,48543.25,10154.0,103425.12,1512300.0,2780997.5,18,80,4.444,0.163,117174.96,425.93,2045267.0,0,6,-1.2088,OrangeMoney
4,USER_13,419,71,5.90,6841.31,2000.0,15136.26,863155.0,2003353.5,74,345,4.662,0.263,9730.25,4.52,125154.0,0,5,-0.0401,OrangeMoney


## 🔗 Step 6 — Merge with Demographics

We join the transaction features (one row per user) with the demographic survey data on `userID`. This gives us a rich dataset combining **behavioural** features (how they use mobile money) with **contextual** features (who they are).

We use a **left join** — every user in the transaction data is kept, even if they have no demographic entry yet.

In [6]:
df_demo_clean = df_demo.copy()

# Standardise column names: lowercase + replace special chars
df_demo_clean.columns = (
    df_demo_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9_]', '_', regex=True)
)

# The id column is 'userid' after normalisation — rename to 'UserId'
# to match df_users (which uses 'UserId' from the pipeline)
if 'userid' in df_demo_clean.columns:
    df_demo_clean = df_demo_clean.rename(columns={'userid': 'UserId'})

# Merge on 'UserId' (left = transaction features, right = demographics)
df_merged = df_users.merge(df_demo_clean, on='UserId', how='left')
print(f'Merged shape: {df_merged.shape}')

# Report how many users have demographic data filled in
gender_col = next((c for c in df_merged.columns
                   if c.lower() == 'gender'), None)
has_demo = df_merged[gender_col].notna().sum() if gender_col else 0
print(f'Users with demographic data: {has_demo}/{len(df_merged)}')
print(df_merged.head(3).to_string())


Merged shape: (16, 29)
Users with demographic data: 14/16
    UserId  total_transactions  months_active  tx_per_month  avg_amount  median_amount  std_amount  total_amount_in  total_amount_out  n_in  n_out  send_receive_ratio  weekend_ratio  avg_balance  min_balance  max_balance  recency_days  n_tx_types  tx_velocity primary_operator age_range gender         occupation            education_level monthly_income_range geographic_zone household_size              primary_mm_use smartphone_ownership
0   USER_1                 147             30          4.90     9872.82         1000.0    17014.06         779704.0          671600.0    45    102               2.267          0.224     14101.95         25.0      90419.0             0           4       0.2187      MobileMoney     46-60      F  Salaried employee  University – Postgraduate        300001-500000       Suburban              7+  Both personal and business                  Yes
1  USER_10                 795             36         22.08 

## 🏷️ Step 7 — Create Activity Label (Prediction Target)

**This is the most important cell for modeling.**

We classify each user as **High**, **Medium**, or **Low** activity based on their `tx_per_month` score.

**Why `tx_per_month`?** It normalises for the fact that some users provided 12 months of data and others only 3. Using raw transaction count would unfairly label someone with 3 months of data as 'low activity' even if they transact frequently.

**Why tertiles (33rd/67th percentile)?** We use the actual distribution of our data to set boundaries, rather than arbitrary numbers. This guarantees roughly balanced classes — important for classification models. This is a **data-driven** decision, which you can defend.

In [7]:
p33 = df_merged['tx_per_month'].quantile(0.33)
p67 = df_merged['tx_per_month'].quantile(0.67)

print(f'Tertile thresholds:')
print(f'  Low activity    : tx_per_month <= {p33:.2f}')
print(f'  Medium activity : {p33:.2f} < tx_per_month <= {p67:.2f}')
print(f'  High activity   : tx_per_month > {p67:.2f}')

def label_activity(tx_rate: float) -> str:
    if tx_rate <= p33:  return 'Low'
    if tx_rate <= p67:  return 'Medium'
    return 'High'

df_merged['activity_label'] = df_merged['tx_per_month'].apply(label_activity)

print('\nClass distribution:')
print(df_merged['activity_label'].value_counts())
print()
print(df_merged['activity_label'].value_counts(normalize=True)
      .mul(100).round(1).astype(str) + '%')


Tertile thresholds:
  Low activity    : tx_per_month <= 6.22
  Medium activity : 6.22 < tx_per_month <= 19.49
  High activity   : tx_per_month > 19.49

Class distribution:
activity_label
Medium    6
Low       5
High      5
Name: count, dtype: int64

activity_label
Medium    37.5%
Low       31.2%
High      31.2%
Name: proportion, dtype: str


## 📦 Step 8 — Outlier Detection & Handling

**Method: IQR (Interquartile Range)**

The IQR is the range between the 25th and 75th percentile of the data. A value is flagged as an outlier if it falls more than 1.5× the IQR below Q1 or above Q3. This is a standard, defensible approach.

**Decision:** We do NOT remove outliers — we **cap** them (also called Winsorization). Removing users entirely would shrink an already small dataset. Capping clips extreme values to the fence while keeping the user in the dataset.

In [8]:
numeric_features = [
    'tx_per_month', 'avg_amount', 'total_amount_in',
    'total_amount_out', 'avg_balance', 'send_receive_ratio',
]

outlier_report = []
df_clean = df_merged.copy()

for col in numeric_features:
    if col not in df_clean.columns: continue
    Q1  = df_clean[col].quantile(0.25)
    Q3  = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    # Cap (Winsorize)
    df_clean[col] = df_clean[col].clip(lower, upper)
    outlier_report.append({
        'Feature': col, 'Q1': round(Q1, 2), 'Q3': round(Q3, 2),
        'IQR': round(IQR, 2), 'Lower fence': round(lower, 2),
        'Upper fence': round(upper, 2), 'Outliers capped': n_outliers,
    })

display(pd.DataFrame(outlier_report))


,Feature,Q1,Q3,IQR,Lower fence,Upper fence,Outliers capped
0,tx_per_month,6.06,20.44,14.37,-15.50,42.00,0
1,avg_amount,4470.97,15405.02,10934.05,-11930.10,31806.10,3
2,total_amount_in,383760.00,2119911.75,1736151.75,-2220467.62,4724139.38,2
3,total_amount_out,880501.25,3877821.75,2997320.50,-3615479.50,8373802.50,3
4,avg_balance,8209.03,37645.35,29436.32,-35945.45,81799.82,2
5,send_receive_ratio,3.27,10.78,7.51,-8.00,22.05,1


## 🔢 Step 9 — Encode Categorical Variables

Machine learning models work with numbers, not text. We convert categorical columns to numbers using two strategies:

- **Ordinal encoding** for variables with a natural order (e.g. Low < Medium < High for income, 18-25 < 26-35 for age). We assign integers that preserve that order.
- **Label encoding** for variables with no meaningful order (e.g. Gender, Geographic zone). We just assign 0, 1, 2…

We keep the original text columns as `_raw` for reference in EDA.

In [9]:
df_encoded = df_clean.copy()

# ── Ordinal mappings (order matters) ──────────────────────────────────
age_map = {
    'Under 18': 0, 'under 18': 0,
    '18-25': 1, '18 - 25': 1, '18–25': 1,
    '26-35': 2, '26 - 35': 2, '26–35': 2,
    '36-45': 3, '36 - 45': 3, '36–45': 3,
    '46-60': 4, '46 - 60': 4, '46–60': 4,
    '60+':   5, '60 +':   5,
}
income_map = {
    '< 50,000 FCFA':          0, 'below 50,000 fcfa': 0,
    '50,000 – 150,000 FCFA':  1, '50,000 - 150,000 fcfa': 1,
    '150,001 – 300,000 FCFA': 2, '150,001 - 300,000 fcfa': 2,
    '300,001 – 500,000 FCFA': 3, '300,001 - 500,000 fcfa': 3,
    'above 500,000 fcfa':     4, 'Above 500,000 FCFA': 4,
    'Prefer not to say':     -1, 'prefer not to say':  -1,
}
household_map = {
    '1': 1, '1 (alone / seul)': 1,
    '2-3': 2, '2 - 3': 2,
    '4-6': 3, '4 - 6': 3,
    '7+':  4,
}

# ── Apply ordinal encodings ───────────────────────────────────────────
for raw_col, enc_col, mapping in [
    ('age_range',    'age_encoded',     age_map),
    ('monthly_income_range', 'income_encoded', income_map),
    ('household_size', 'household_encoded', household_map),
]:
    if raw_col in df_encoded.columns:
        df_encoded[f'{raw_col}_raw'] = df_encoded[raw_col]
        df_encoded[enc_col] = (
            df_encoded[raw_col].astype(str)
            .str.strip()
            .map(mapping)
        )

# ── Label encode remaining categoricals ───────────────────────────────
from sklearn.preprocessing import LabelEncoder
label_cols = ['gender', 'occupation', 'education_level',
              'geographic_zone', 'primary_mm_use']
le = LabelEncoder()
for col in label_cols:
    if col in df_encoded.columns:
        df_encoded[f'{col}_raw'] = df_encoded[col]
        df_encoded[col] = le.fit_transform(
            df_encoded[col].astype(str).str.strip()
        )

# ── Encode the target label ───────────────────────────────────────────
activity_map = {'Low': 0, 'Medium': 1, 'High': 2}
df_encoded['activity_label_encoded'] = \
    df_encoded['activity_label'].map(activity_map)

print('✅ Encoding complete')
display(df_encoded.dtypes)


✅ Encoding complete


UserId                          str
total_transactions            int64
months_active                 int64
tx_per_month                float64
avg_amount                  float64
median_amount               float64
std_amount                  float64
total_amount_in             float64
total_amount_out            float64
n_in                          int64
n_out                         int64
send_receive_ratio          float64
weekend_ratio               float64
avg_balance                 float64
min_balance                 float64
max_balance                 float64
recency_days                  int64
n_tx_types                    int64
tx_velocity                 float64
primary_operator                str
age_range                       str
gender                        int64
occupation                    int64
education_level               int64
monthly_income_range            str
geographic_zone               int64
household_size                  str
primary_mm_use              

## 📊 Step 10 — Summary Statistics (Before vs After)

Good data cleaning reports show numbers, not just words. This table goes directly into your **Data Cleaning Report**.

In [10]:
numeric_cols = [
    'total_transactions', 'tx_per_month', 'avg_amount',
    'total_amount_in', 'total_amount_out', 'avg_balance',
    'send_receive_ratio', 'weekend_ratio',
]
existing = [c for c in numeric_cols if c in df_encoded.columns]

print('=== Summary Statistics (Cleaned User-Level Dataset) ===')
stats = df_encoded[existing].describe().round(2)
display(stats)

print('\n=== Activity Label Distribution ===')
display(
    df_encoded['activity_label'].value_counts()
    .to_frame(name='Count')
    .assign(Percentage=lambda x: (x['Count']/len(df_encoded)*100).round(1))
)


=== Summary Statistics (Cleaned User-Level Dataset) ===


,total_transactions,tx_per_month,avg_amount,total_amount_in,total_amount_out,avg_balance,send_receive_ratio,weekend_ratio
count,16.00,16.00,16.00,16.00,16.00,16.00,16.00,16.00
mean,485.75,13.76,12195.58,1515941.17,3032962.06,27357.01,7.04,0.27
std,394.84,9.51,10805.61,1536638.88,2950896.60,26928.62,5.67,0.04
min,54.00,2.25,2562.96,59450.00,66100.00,3024.69,1.00,0.16
25%,147.00,6.06,4470.97,383760.00,880501.25,8209.03,3.27,0.25
50%,427.50,10.56,7401.60,995222.50,2117365.25,15703.92,4.89,0.27
75%,740.25,20.44,15405.02,2119911.75,3877821.75,37645.35,10.78,0.29
max,1519.00,31.65,31806.10,4724139.38,8373802.50,81799.82,22.05,0.37



=== Activity Label Distribution ===


,Count,Percentage
activity_label,,
Medium,6,37.5
Low,5,31.2
High,5,31.2


## 💾 Step 11 — Export Cleaned Dataset

We save the final cleaned dataset. This file is the single input for both the EDA notebook and the Modeling notebook.

In [11]:
df_encoded.to_csv(CLEANED_FILE, index=False, encoding='utf-8-sig')
print(f'✅ Saved: {CLEANED_FILE}')
print(f'   Shape: {df_encoded.shape[0]} users × {df_encoded.shape[1]} columns')
print()

# Quick column inventory
print('Columns in cleaned dataset:')
for col in df_encoded.columns:
    print(f'  {col:35s} {str(df_encoded[col].dtype):10s} '
          f'  nulls: {df_encoded[col].isna().sum()}')


✅ Saved: C:\Users\joelf\Documents\GitHub\mobile_money_analysis\2_Data_Cleaning\cleaned_data.csv
   Shape: 16 users × 42 columns

Columns in cleaned dataset:
  UserId                              str          nulls: 0
  total_transactions                  int64        nulls: 0
  months_active                       int64        nulls: 0
  tx_per_month                        float64      nulls: 0
  avg_amount                          float64      nulls: 0
  median_amount                       float64      nulls: 0
  std_amount                          float64      nulls: 0
  total_amount_in                     float64      nulls: 0
  total_amount_out                    float64      nulls: 0
  n_in                                int64        nulls: 0
  n_out                               int64        nulls: 0
  send_receive_ratio                  float64      nulls: 0
  weekend_ratio                       float64      nulls: 0
  avg_balance                         float64      nulls: 0
  m